# Two-stage HB diagnostic notebook

This notebook lets you inspect one portfolio year without running `run_main.py`.

It checks:
- the estimation window used for a chosen year
- the external CFO model inputs
- `CFO_lead1_pred_scaled` after external prediction
- whether NaNs / infs remain before the accrual model
- whether the accrual model can be built and whether its initial point is valid


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pymc as pm


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'data').exists() and (p / 'results').exists():
            return p
    raise FileNotFoundError("Could not find project root containing 'data' and 'results'.")


PROJECT_ROOT = find_project_root()
SHARED_DIR = PROJECT_ROOT / 'modelling' / 'shared'
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SHARED_DIR   =', SHARED_DIR)


In [ ]:
from hb_shared_utils import (
    compute_wca,
    build_regressors,
    assign_indices,
    mark_usable,
    build_estimation_window,
)

from uncertainty_model_hb import (
    fit_external_cfo_ar1_model,
    predict_cfo_lead_for_portfolio_rows,
    build_hb_accrual_model_fixed_lead,
)


In [ ]:
# --- Settings ---
INPUT_CSV = PROJECT_ROOT / 'results' / 'extraction_static' / 'prepared_step2_input.csv'

PORT_YEAR = 2022
MIN_TRAIN_YEARS = 3
MAX_TRAIN_YEARS = 5

# Smaller CFO model for diagnostics
RANDOM_SEED = 42
CFO_DRAWS = 300
CFO_TUNE = 500
N_CHAINS = 4
TARGET_ACCEPT = 0.95
CFO_PREDICTION_MODE = 'mean'   # 'mean' or 'draw'

print('INPUT_CSV =', INPUT_CSV)
print('PORT_YEAR =', PORT_YEAR)


In [ ]:
# --- Load and prepare panel exactly like Step 2 ---
data = pd.read_csv(INPUT_CSV)
data = compute_wca(data)
data = build_regressors(data, include_lead=True)
data = mark_usable(data)
data, firm_map, sector_map, firm_sector = assign_indices(data)

print('Panel shape:', data.shape)
print('Years:', int(data['Year'].min()), 'to', int(data['Year'].max()))
print('Usable rows:', int(data['usable'].sum()))
print('Unique firms:', int(data['Ticker'].nunique()))
print('CFO_lead1_scaled non-null:', int(data['CFO_lead1_scaled'].notna().sum()))


In [ ]:
# --- Build the exact estimation window for the chosen portfolio year ---
window_df = build_estimation_window(
    data,
    PORT_YEAR,
    min_train_years=MIN_TRAIN_YEARS,
    max_train_years=MAX_TRAIN_YEARS,
    include_portfolio_year=True,
)

if window_df is None:
    raise ValueError(f'No estimation window could be built for PORT_YEAR={PORT_YEAR}')

print('window_df shape:', window_df.shape)
print('window years:', sorted(window_df['Year'].dropna().astype(int).unique().tolist()))
print('unique firms in window:', int(window_df['Ticker'].nunique()))
print('portfolio-year rows:', int(window_df['is_portfolio_year'].sum()))
print('training rows:', int((~window_df['is_portfolio_year']).sum()))

display(window_df[['Ticker','Year','is_portfolio_year','CFO_scaled','CFO_lead1_scaled','WCA_scaled']].head(20))


In [ ]:
# --- Missingness before external CFO prediction ---
pre_cols = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

print('Missing values in raw window:')
display(window_df[pre_cols].isna().sum().to_frame('n_missing'))

bad_pre = window_df[pre_cols].isna().any(axis=1)
print('Rows with any missing among pre_cols:', int(bad_pre.sum()))
display(window_df.loc[bad_pre, ['Ticker','Year','is_portfolio_year'] + pre_cols].head(20))


In [ ]:
# --- Fit external CFO model only ---
cfo_info, cfo_trace = fit_external_cfo_ar1_model(
    window_df=window_df,
    random_seed=RANDOM_SEED + 10_000 + int(PORT_YEAR),
    draws=CFO_DRAWS,
    tune=CFO_TUNE,
    chains=N_CHAINS,
    target_accept=TARGET_ACCEPT,
)

print('External CFO model fitted.')
print('n_ar1_obs =', cfo_info.get('n_ar1_obs'))
print('n_latent  =', cfo_info.get('n_latent', 'not found'))
print('keys      =', sorted(cfo_info.keys()))


In [ ]:
# --- Predict CFO_{t+1} externally ---
window_df_fixed = predict_cfo_lead_for_portfolio_rows(
    window_df=window_df,
    cfo_info=cfo_info,
    cfo_trace=cfo_trace,
    prediction_mode=CFO_PREDICTION_MODE,
)

check_cols = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_scaled',
    'CFO_lead1_pred_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

print('Missing values after external CFO prediction:')
display(window_df_fixed[check_cols].isna().sum().to_frame('n_missing'))

finite_flags = np.isfinite(window_df_fixed[check_cols].to_numpy(dtype=float, copy=True))
print('All finite in check_cols:', bool(finite_flags.all()))

bad_post = window_df_fixed[check_cols].isna().any(axis=1) | (~np.isfinite(window_df_fixed[check_cols].to_numpy(dtype=float, copy=True)).all(axis=1))
print('Rows with any missing/non-finite after prediction:', int(bad_post.sum()))
display(window_df_fixed.loc[bad_post, ['Ticker','Year','is_portfolio_year'] + check_cols].head(50))


In [ ]:
# --- Compare original vs predicted CFO lead on portfolio rows ---
portfolio_rows = window_df_fixed['is_portfolio_year'].fillna(False)
compare_cols = ['Ticker','Year','CFO_scaled','CFO_lead1_scaled','CFO_lead1_pred_scaled']
display(window_df_fixed.loc[portfolio_rows, compare_cols].head(30))


In [ ]:
# --- Optional safeguard used only for diagnostics ---
# Drop rows with missing regressors and inspect how many would be removed.
needed = [
    'WCA_scaled',
    'CFO_lag1_scaled',
    'CFO_scaled',
    'CFO_lead1_pred_scaled',
    'dREV_scaled',
    'PPE_scaled',
]

diag_bad = window_df_fixed[needed].isna().any(axis=1)
print('Rows that would be dropped before accrual model:', int(diag_bad.sum()))

window_df_model = window_df_fixed.loc[~diag_bad].copy()
print('window_df_model shape:', window_df_model.shape)
display(window_df_model[['Ticker','Year','is_portfolio_year'] + needed].head(20))


In [ ]:
# --- Build accrual model only ---
model, trace_info = build_hb_accrual_model_fixed_lead(window_df_model, firm_sector)
print('Accrual model built.')
print(trace_info)


In [ ]:
# --- Check initial point before sampling ---
with model:
    start = model.initial_point()
    print('Initial point keys:', list(start.keys())[:10], '...')

    try:
        check = model.check_start_vals(start)
        print('Initial point check passed.')
        print(check)
    except Exception as e:
        print('Initial point check failed:')
        print(type(e).__name__, e)
        print('\nRunning model.debug() ...')
        model.debug()


In [ ]:
# --- Tiny smoke-test sample if the initial point is OK ---
# Keep this very small for diagnostics.
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    with model:
        trace = pm.sample(
            draws=50,
            tune=50,
            chains=2,
            target_accept=0.9,
            random_seed=RANDOM_SEED + int(PORT_YEAR),
            return_inferencedata=True,
            progressbar=True,
        )
    print(trace)
else:
    print('RUN_SMOKE_TEST = False, so no accrual sampling was run.')
